# Kreol Morisien Hallucination Pilot — Direct Run

This notebook uses the attached, already reviewed `pilot_gold_review_completed.xlsx` directly.

**There is no human-review gate.** It does not generate another gold-review queue and does not stop for approval. It runs the approved cases, stores model outputs and uncertainty signals, applies provisional automatic failure categorization, compares failure patterns with dataset characteristics, and exports candidate benchmark-dimension evidence.

The fourth model is `HuggingFaceTB/SmolLM3-3B`; Gemma is not used.

## 1. Install dependencies once on RunPod

In [1]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    import sys, subprocess
    packages = [
        "transformers>=4.51.0", "accelerate>=1.2.0", "huggingface_hub>=0.27.0",
        "safetensors>=0.4.5", "sentencepiece>=0.2.0", "protobuf>=5.0",
        "pandas>=2.2", "pyarrow>=17.0", "openpyxl>=3.1", "psutil>=6.0",
        "matplotlib>=3.9", "scikit-learn>=1.5", "seaborn>=0.13"
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *packages])
    print("Restart the kernel now, then run from the top.")
else:
    print("Installation disabled. Set INSTALL_DEPENDENCIES=True only once if packages are missing.")

Installation disabled. Set INSTALL_DEPENDENCIES=True only once if packages are missing.


## 2. Configuration — models run without a review gate

In [2]:
from pathlib import Path
import os, json, re, time, gc, math, hashlib, platform, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime, timezone
import pandas as pd
import numpy as np

NOTEBOOK_VERSION = "1.0-direct-reviewed-input"
SEED = "KM-PILOT-20260915"
INPUT_XLSX = Path(os.getenv("KM_REVIEWED_XLSX", "pilot_gold_review_completed.xlsx"))
OUTPUT_DIR = Path(os.getenv("KM_PILOT_OUTPUT", "km_direct_pilot_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODELS = True
RUN_AUTOMATED_JUDGE = True
RUN_HIDDEN_STATE_ANALYSIS = False
RESUME = True
CASE_LIMIT = None       # None = all approved rows in the workbook; use an integer only for a technical test
MODEL_KEYS = ["qwen_instruct", "qwen_thinking", "phi4_mini", "smollm3"]
JUDGE_MODEL_KEY = "qwen_instruct"

LOAD_STRATEGY = "AUTO" # AUTO | BF16 | FP16 | Q4
MAX_INPUT_TOKENS = 4096
TOP_K = 5
HF_TOKEN = os.getenv("HF_TOKEN")
HF_HOME = os.getenv("HF_HOME")

MODEL_REGISTRY = {
    "qwen_instruct": {"model_id":"Qwen/Qwen3-4B-Instruct-2507", "family":"Qwen", "reasoning":"non-thinking"},
    "qwen_thinking": {"model_id":"Qwen/Qwen3-4B-Thinking-2507", "family":"Qwen", "reasoning":"thinking"},
    "phi4_mini": {"model_id":"microsoft/Phi-4-mini-instruct", "family":"Phi", "reasoning":"instruct"},
    "smollm3": {"model_id":"HuggingFaceTB/SmolLM3-3B", "family":"SmolLM", "reasoning":"dual-mode"},
}

MAX_NEW_TOKENS_BY_FAMILY = {
    "answerability_abstention": 96, "attribution": 96, "evidence_grounded_qa": 160,
    "exact_list_extraction": 128, "explicit_claim_extraction": 256,
    "conversational_grounding": 192, "reply_relation_classification": 128,
    "faithful_summarization": 256, "reference_resolution": 128,
    "evidence_supported_inference": 192, "stance_uncertainty_summary": 256,
    "placeholder_continuity": 160, "missing_context_abstention": 96,
    "source_overreach_probe": 128,
}
DEFAULT_MAX_NEW_TOKENS = 192
print("Input:", INPUT_XLSX)
print("Output:", OUTPUT_DIR.resolve())
print("Models:", [MODEL_REGISTRY[k]["model_id"] for k in MODEL_KEYS])

Input: pilot_gold_review_completed.xlsx
Output: /workspace/mauritian_creole_nlp/benchmark_test/km_direct_pilot_outputs
Models: ['Qwen/Qwen3-4B-Instruct-2507', 'Qwen/Qwen3-4B-Thinking-2507', 'microsoft/Phi-4-mini-instruct', 'HuggingFaceTB/SmolLM3-3B']


## 3. Runtime and GPU preflight

In [3]:
def command_output(cmd):
    try: return subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT, timeout=30)
    except Exception as exc: return f"UNAVAILABLE: {type(exc).__name__}: {exc}"

runtime = {
    "utc": datetime.now(timezone.utc).isoformat(), "python": platform.python_version(),
    "platform": platform.platform(), "input_exists": INPUT_XLSX.exists(),
    "disk_free_gb": round(shutil.disk_usage(OUTPUT_DIR).free/1e9,2),
    "nvidia_smi": command_output(["nvidia-smi"]) if shutil.which("nvidia-smi") else "NOT_FOUND",
}
try:
    import torch
    runtime.update({"torch":torch.__version__, "cuda_available":torch.cuda.is_available(),
                    "cuda":torch.version.cuda, "gpu_count":torch.cuda.device_count(),
                    "gpus":[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]})
except Exception as exc:
    runtime["torch_error"] = str(exc)
(OUTPUT_DIR/"environment_report.json").write_text(json.dumps(runtime,indent=2),encoding="utf-8")
print({k:v for k,v in runtime.items() if k != "nvidia_smi"})
if RUN_MODELS and not runtime.get("cuda_available"):
    raise RuntimeError("RUN_MODELS=True requires a CUDA GPU. Use a GPU RunPod or set RUN_MODELS=False for analysis-only execution.")

{'utc': '2026-09-18T04:48:44.716161+00:00', 'python': '3.12.3', 'platform': 'Linux-6.8.0-100-generic-x86_64-with-glibc2.39', 'input_exists': True, 'disk_free_gb': 909666.88, 'torch': '2.8.0+cu128', 'cuda_available': True, 'cuda': '12.8', 'gpu_count': 1, 'gpus': ['NVIDIA GeForce RTX 4090']}


## 4. Utilities and reproducibility

In [4]:
def sha256_file(path, chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(chunk),b""): h.update(block)
    return h.hexdigest()
def normalize_text(x): return re.sub(r"\s+"," ",str(x or "").strip().lower())
def truthy(x): return str(x).strip().upper() in {"TRUE","YES","Y","1"}
def safe_json(x):
    if isinstance(x,(dict,list)): return x
    if x is None or (isinstance(x,float) and math.isnan(x)): return None
    try: return json.loads(str(x))
    except Exception: return str(x)
def append_jsonl(path,row):
    with open(path,"a",encoding="utf-8") as f: f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n")
def load_jsonl(path):
    if not Path(path).exists(): return []
    return [json.loads(x) for x in Path(path).read_text(encoding="utf-8").splitlines() if x.strip()]
def tokenize_words(text): return re.findall(r"[A-Za-zÀ-ÿ0-9_]+", normalize_text(text))
def token_f1(pred,gold):
    p=Counter(tokenize_words(pred)); g=Counter(tokenize_words(gold))
    overlap=sum((p&g).values())
    if not p or not g: return float(p==g)
    precision=overlap/sum(p.values()); recall=overlap/sum(g.values())
    return 0 if precision+recall==0 else 2*precision*recall/(precision+recall)

def choose_load_strategy():
    if LOAD_STRATEGY != "AUTO": return LOAD_STRATEGY
    import torch
    if not torch.cuda.is_available(): return "CPU"
    total=torch.cuda.get_device_properties(0).total_memory/1024**3
    return "BF16" if total >= 14 and torch.cuda.is_bf16_supported() else ("FP16" if total >= 14 else "Q4")
EFFECTIVE_LOAD_STRATEGY=choose_load_strategy() if RUN_MODELS else "NOT_APPLICABLE"
print("Effective load strategy:",EFFECTIVE_LOAD_STRATEGY)

Effective load strategy: BF16


In [5]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


## 5. Load the already reviewed workbook directly

In [6]:
if not INPUT_XLSX.exists(): raise FileNotFoundError(f"Reviewed workbook not found: {INPUT_XLSX}")
RAW = pd.read_excel(INPUT_XLSX, sheet_name="gold_review_queue", engine="openpyxl")
required=["case_id","thread_id","task_class","task_family","supplied_context","question_or_instruction","approved_for_pilot"]
missing=[c for c in required if c not in RAW.columns]
if missing: raise ValueError(f"Missing columns: {missing}")
APPROVED = RAW[RAW["approved_for_pilot"].map(truthy)].copy()
if "drop_case_yes_no" in APPROVED.columns:
    APPROVED = APPROVED[~APPROVED["drop_case_yes_no"].map(truthy)].copy()
APPROVED = APPROVED.sort_values("case_id").reset_index(drop=True)
if CASE_LIMIT is not None: APPROVED = APPROVED.head(int(CASE_LIMIT)).copy()
if APPROVED.empty: raise ValueError("No approved rows found in the reviewed workbook.")
APPROVED["effective_gold_answer"] = APPROVED.apply(lambda r: r.get("authored_or_corrected_gold_answer") if pd.notna(r.get("authored_or_corrected_gold_answer")) else r.get("machine_provisional_answer"),axis=1)
APPROVED["effective_evidence"] = APPROVED.apply(lambda r: r.get("corrected_evidence_display_ids") if pd.notna(r.get("corrected_evidence_display_ids")) else r.get("machine_gold_evidence"),axis=1)
APPROVED["effective_answerability"] = APPROVED.apply(lambda r: r.get("confirmed_answerability") if pd.notna(r.get("confirmed_answerability")) else r.get("machine_answerability_label"),axis=1)
APPROVED.to_parquet(OUTPUT_DIR/"approved_cases.parquet",index=False)
summary={"workbook_sha256":sha256_file(INPUT_XLSX),"rows_in_workbook":len(RAW),"approved_cases_loaded":len(APPROVED),"task_classes":APPROVED.task_class.value_counts().to_dict(),"task_families":APPROVED.task_family.value_counts().to_dict(),"strata":APPROVED.stratum.value_counts(dropna=False).to_dict() if "stratum" in APPROVED else {}}
(OUTPUT_DIR/"reviewed_input_profile.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
print(json.dumps(summary,indent=2,default=str))

{
  "workbook_sha256": "befe1b547d0bd1d8280f4f1190e02a48387fc23b9ef175120be7523bd8c59934",
  "rows_in_workbook": 951,
  "approved_cases_loaded": 951,
  "task_classes": {
    "PIPELINE_CONTROL": 516,
    "LINGUISTIC_DIAGNOSTIC": 435
  },
  "task_families": {
    "answerability_abstention": 240,
    "attribution": 109,
    "evidence_grounded_qa": 107,
    "explicit_claim_extraction": 96,
    "exact_list_extraction": 60,
    "conversational_grounding": 59,
    "reply_relation_classification": 59,
    "faithful_summarization": 49,
    "reference_resolution": 48,
    "evidence_supported_inference": 43,
    "stance_uncertainty_summary": 43,
    "placeholder_continuity": 34,
    "missing_context_abstention": 3,
    "source_overreach_probe": 1
  },
  "strata": {
    "depth2": 379,
    "depth1": 315,
    "depth0": 257
  }
}


## 6. Corpus/task pre-analysis from the reviewed cases

In [7]:
def case_features(r):
    context=str(r.get("supplied_context") or "")
    placeholders=re.findall(r"PERSON_\d+",context)
    turns=re.findall(r"\[(C|R)\d+",context)
    return {
        "case_id":r["case_id"],"thread_id":r["thread_id"],"post_id":r.get("post_id"),
        "stratum":r.get("stratum"),"task_class":r.get("task_class"),"task_family":r.get("task_family"),"task_variant":r.get("task_variant"),
        "cand_language_profile":r.get("cand_language_profile"),"validated_language_profile":r.get("validated_language_profile"),
        "validated_code_switching":r.get("validated_code_switching"),"reference_ambiguity":r.get("reference_ambiguity"),
        "exophoric_reference":r.get("exophoric_reference"),"local_cultural_dependence":r.get("local_cultural_dependence"),
        "evidence_sufficiency":r.get("evidence_sufficiency"),"fragmentary_confirmed":r.get("fragmentary_confirmed"),"ambiguity_flag":r.get("ambiguity_flag"),
        "answerability":r.get("effective_answerability"),"context_chars":len(context),"context_tokens_ws":len(context.split()),
        "turn_markers":len(turns),"has_url":bool(re.search(r"https?://",context)),"has_placeholder":bool(placeholders),
        "placeholder_recurrence":len(placeholders)>=2,"placeholder_count":len(placeholders),
        "has_emoji":any(ord(ch)>10000 for ch in context),"expressive_punctuation":bool(re.search(r"([!?\.])\1{2,}",context)),
        "repeated_characters":bool(re.search(r"(.)\1{3,}",context,re.I)),
    }
FEATURES=pd.DataFrame([case_features(r) for _,r in APPROVED.iterrows()])
FEATURES.to_parquet(OUTPUT_DIR/"case_features.parquet",index=False)
for col in ["task_family","task_class","stratum","cand_language_profile","answerability"]:
    if col in FEATURES: print("\n",col,"\n",FEATURES[col].value_counts(dropna=False).head(20))


 task_family 
 task_family
answerability_abstention         240
attribution                      109
evidence_grounded_qa             107
explicit_claim_extraction         96
exact_list_extraction             60
conversational_grounding          59
reply_relation_classification     59
faithful_summarization            49
reference_resolution              48
evidence_supported_inference      43
stance_uncertainty_summary        43
placeholder_continuity            34
missing_context_abstention         3
source_overreach_probe             1
Name: count, dtype: int64

 task_class 
 task_class
PIPELINE_CONTROL         516
LINGUISTIC_DIAGNOSTIC    435
Name: count, dtype: int64

 stratum 
 stratum
depth2    379
depth1    315
depth0    257
Name: count, dtype: int64

 cand_language_profile 
 cand_language_profile
KM+FR        576
KM           123
KM+FR+EN      95
FR            73
NO_MARKER     26
KM+EN         25
EN            24
FR+EN          9
Name: count, dtype: int64

 answerability 
 an

## 7. Model registry and live checks

In [8]:
from huggingface_hub import HfApi
api=HfApi(token=HF_TOKEN)
registry=[]
for key in MODEL_KEYS:
    item=dict(MODEL_REGISTRY[key]); item["key"]=key
    try:
        info=api.model_info(item["model_id"])
        item.update({"status":"VERIFIED_HUB_LIVE","revision":info.sha,"gated":info.gated,"private":info.private})
    except Exception as exc:
        item.update({"status":"UNAVAILABLE","error":f"{type(exc).__name__}: {exc}"})
    registry.append(item)
REGISTRY=pd.DataFrame(registry)
REGISTRY.to_csv(OUTPUT_DIR/"model_registry.csv",index=False)
display(REGISTRY)
if RUN_MODELS and (REGISTRY.status!="VERIFIED_HUB_LIVE").any():
    raise RuntimeError("At least one model repository is unavailable. See model_registry.csv; no silent substitution was made.")

,model_id,family,reasoning,key,status,revision,gated,private
0,Qwen/Qwen3-4B-Instruct-2507,Qwen,non-thinking,qwen_instruct,VERIFIED_HUB_LIVE,cdbee75f17c01a7cc42f958dc650907174af0554,False,False
1,Qwen/Qwen3-4B-Thinking-2507,Qwen,thinking,qwen_thinking,VERIFIED_HUB_LIVE,768f209d9ea81521153ed38c47d515654e938aea,False,False
2,microsoft/Phi-4-mini-instruct,Phi,instruct,phi4_mini,VERIFIED_HUB_LIVE,cfbefacb99257ffa30c83adab238a50856ac3083,False,False
3,HuggingFaceTB/SmolLM3-3B,SmolLM,dual-mode,smollm3,VERIFIED_HUB_LIVE,a07cc9a04f16550a088caea529712d1d335b0ac1,False,False


## 8. Prompt construction

In [9]:
SYSTEM_PROMPT = """You are being evaluated on evidence-grounded understanding of informal Kreol Morisien social-media conversations. Use only the supplied conversation. Do not add external facts. Do not infer identities behind PERSON placeholders. Preserve uncertainty. Follow the requested output format. If the evidence is insufficient, use INSUFFICIENT_EVIDENCE."""
def make_messages(r):
    task=str(r["question_or_instruction"])
    context=str(r["supplied_context"])
    user=f"Conversation:\n{context}\n\nTask:\n{task}"
    return [{"role":"system","content":SYSTEM_PROMPT},{"role":"user","content":user}]
def max_new_tokens_for(r): return int(MAX_NEW_TOKENS_BY_FAMILY.get(str(r.get("task_family")),DEFAULT_MAX_NEW_TOKENS))
print(make_messages(APPROVED.iloc[0]))

[{'role': 'system', 'content': 'You are being evaluated on evidence-grounded understanding of informal Kreol Morisien social-media conversations. Use only the supplied conversation. Do not add external facts. Do not infer identities behind PERSON placeholders. Preserve uncertainty. Follow the requested output format. If the evidence is insufficient, use INSUFFICIENT_EVIDENCE.'}, {'role': 'user', 'content': 'Conversation:\n[C1] Blessé par balle tout hein ?! Wow ! Et pourtant, tout va bien à Maurice non ? Le taux de criminalité n\'a-t-il pas baissé ???!!!!!\n[R1 -> C1] PERSON_000004 t\'emballe pas Majo!\n[R2 -> R1] PERSON_000005 Oh non, loin de là\n\nTask:\nHow many turns does this conversation contain? Reply with JSON only: {"answer": <integer>}. If the evidence is insufficient reply {"answer": "INSUFFICIENT_EVIDENCE"}.'}]


## 9. Sequential model loading

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_bundle(model_id):
    common={"token":HF_TOKEN,"cache_dir":HF_HOME,"trust_remote_code":False}
    kwargs=dict(common)
    if EFFECTIVE_LOAD_STRATEGY=="BF16": kwargs.update(torch_dtype=torch.bfloat16,device_map="auto")
    elif EFFECTIVE_LOAD_STRATEGY=="FP16": kwargs.update(torch_dtype=torch.float16,device_map="auto")
    elif EFFECTIVE_LOAD_STRATEGY=="Q4":
        kwargs.update(device_map="auto",quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_compute_dtype=torch.bfloat16,bnb_4bit_quant_type="nf4"))
    elif EFFECTIVE_LOAD_STRATEGY=="CPU": kwargs.update(torch_dtype=torch.float32,device_map={"":"cpu"})
    else: raise ValueError(EFFECTIVE_LOAD_STRATEGY)
    tokenizer=AutoTokenizer.from_pretrained(model_id,**common)
    model=AutoModelForCausalLM.from_pretrained(model_id,**kwargs)
    model.eval()
    if tokenizer.pad_token_id is None: tokenizer.pad_token_id=tokenizer.eos_token_id
    return tokenizer,model

def unload_bundle(tokenizer,model):
    del tokenizer,model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

## 10. Inference, logits and uncertainty

In [11]:
RESULTS_PATH=OUTPUT_DIR/"model_outputs.jsonl"
TOKEN_PATH=OUTPUT_DIR/"token_uncertainty.jsonl"
existing=load_jsonl(RESULTS_PATH) if RESUME else []
completed={(x.get("case_id"),x.get("model_id"),x.get("prompt_hash"),x.get("load_strategy")) for x in existing if x.get("status")=="OK"}

def generate_case(tokenizer,model,row,model_id,revision):
    messages=make_messages(row)
    rendered=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True) if getattr(tokenizer,"chat_template",None) else SYSTEM_PROMPT+"\n\n"+messages[-1]["content"]+"\n\nAnswer:"
    prompt_hash=hashlib.sha256(rendered.encode()).hexdigest()
    signature=(row["case_id"],model_id,prompt_hash,EFFECTIVE_LOAD_STRATEGY)
    if RESUME and signature in completed: return None,[]
    enc=tokenizer(rendered,return_tensors="pt",truncation=True,max_length=MAX_INPUT_TOKENS)
    device=next(model.parameters()).device; enc={k:v.to(device) for k,v in enc.items()}
    start=time.time()
    with torch.inference_mode():
        out=model.generate(**enc,max_new_tokens=max_new_tokens_for(row),do_sample=False,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.pad_token_id)
    gen=out.sequences[0,enc["input_ids"].shape[1]:]
    text=tokenizer.decode(gen,skip_special_tokens=True)
    token_rows=[]
    for i,(tid,score) in enumerate(zip(gen.tolist(),out.scores)):
        logits=score[0].float(); logp=torch.log_softmax(logits,-1); p=torch.softmax(logits,-1)
        vals,idx=torch.topk(p,k=min(TOP_K,p.shape[-1]))
        token_rows.append({"case_id":row["case_id"],"thread_id":row["thread_id"],"model_id":model_id,"revision":revision,"token_index":i,"token_id":tid,"token":tokenizer.decode([tid]),"probability":float(p[tid].cpu()),"logprob":float(logp[tid].cpu()),"entropy":float((-(p*logp).sum()).cpu()),"top_k":[{"token":tokenizer.decode([int(j)]),"probability":float(v)} for j,v in zip(idx.cpu(),vals.cpu())]})
    result={"status":"OK","case_id":row["case_id"],"thread_id":row["thread_id"],"post_id":row.get("post_id"),"task_class":row.get("task_class"),"task_family":row.get("task_family"),"task_variant":row.get("task_variant"),"model_id":model_id,"revision":revision,"load_strategy":EFFECTIVE_LOAD_STRATEGY,"prompt_hash":prompt_hash,"input_token_ids":enc["input_ids"][0].cpu().tolist(),"attention_mask":enc["attention_mask"][0].cpu().tolist(),"generated_token_ids":gen.cpu().tolist(),"raw_output":text,"input_tokens":int(enc["input_ids"].shape[1]),"output_tokens":int(len(gen)),"max_new_tokens":max_new_tokens_for(row),"duration_seconds":round(time.time()-start,3),"generated_utc":datetime.now(timezone.utc).isoformat()}
    return result,token_rows

## 11. Run all approved cases on all four models

In [ ]:
if RUN_MODELS:
    revisions=dict(zip(REGISTRY.model_id,REGISTRY.revision))
    for key in MODEL_KEYS:
        model_id=MODEL_REGISTRY[key]["model_id"]
        print("\nLoading",model_id)
        tokenizer,model=load_bundle(model_id)
        try:
            for n,(_,row) in enumerate(APPROVED.iterrows(),1):
                try:
                    result,tokens=generate_case(tokenizer,model,row,model_id,revisions.get(model_id))
                    if result:
                        append_jsonl(RESULTS_PATH,result)
                        for tr in tokens: append_jsonl(TOKEN_PATH,tr)
                    if n%25==0 or n==len(APPROVED): print(model_id,n,"/",len(APPROVED))
                except Exception as exc:
                    append_jsonl(RESULTS_PATH,{"status":"ERROR","case_id":row["case_id"],"thread_id":row["thread_id"],"model_id":model_id,"error":f"{type(exc).__name__}: {exc}","generated_utc":datetime.now(timezone.utc).isoformat()})
        finally:
            unload_bundle(tokenizer,model)
else:
    print("RUN_MODELS=False: using any existing outputs only.")


Loading Qwen/Qwen3-4B-Instruct-2507


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen/Qwen3-4B-Instruct-2507 25 / 951
Qwen/Qwen3-4B-Instruct-2507 50 / 951
Qwen/Qwen3-4B-Instruct-2507 75 / 951
Qwen/Qwen3-4B-Instruct-2507 100 / 951
Qwen/Qwen3-4B-Instruct-2507 125 / 951
Qwen/Qwen3-4B-Instruct-2507 150 / 951
Qwen/Qwen3-4B-Instruct-2507 175 / 951
Qwen/Qwen3-4B-Instruct-2507 200 / 951
Qwen/Qwen3-4B-Instruct-2507 225 / 951
Qwen/Qwen3-4B-Instruct-2507 250 / 951
Qwen/Qwen3-4B-Instruct-2507 275 / 951
Qwen/Qwen3-4B-Instruct-2507 300 / 951
Qwen/Qwen3-4B-Instruct-2507 325 / 951
Qwen/Qwen3-4B-Instruct-2507 350 / 951
Qwen/Qwen3-4B-Instruct-2507 375 / 951
Qwen/Qwen3-4B-Instruct-2507 400 / 951
Qwen/Qwen3-4B-Instruct-2507 425 / 951
Qwen/Qwen3-4B-Instruct-2507 450 / 951
Qwen/Qwen3-4B-Instruct-2507 475 / 951
Qwen/Qwen3-4B-Instruct-2507 500 / 951
Qwen/Qwen3-4B-Instruct-2507 525 / 951
Qwen/Qwen3-4B-Instruct-2507 550 / 951
Qwen/Qwen3-4B-Instruct-2507 575 / 951
Qwen/Qwen3-4B-Instruct-2507 600 / 951
Qwen/Qwen3-4B-Instruct-2507 625 / 951
Qwen/Qwen3-4B-Instruct-2507 650 / 951
Qwen/Qwen3-4B-I

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


HuggingFaceTB/SmolLM3-3B 25 / 951
HuggingFaceTB/SmolLM3-3B 50 / 951
HuggingFaceTB/SmolLM3-3B 75 / 951
HuggingFaceTB/SmolLM3-3B 100 / 951
HuggingFaceTB/SmolLM3-3B 125 / 951
HuggingFaceTB/SmolLM3-3B 150 / 951
HuggingFaceTB/SmolLM3-3B 175 / 951
HuggingFaceTB/SmolLM3-3B 200 / 951
HuggingFaceTB/SmolLM3-3B 225 / 951


## 12. Mechanical and evidence-grounding diagnostics

In [ ]:
OUTPUTS=pd.DataFrame(load_jsonl(RESULTS_PATH)) if RESULTS_PATH.exists() else pd.DataFrame()
TOKENS=pd.DataFrame(load_jsonl(TOKEN_PATH)) if TOKEN_PATH.exists() else pd.DataFrame()
if OUTPUTS.empty: raise RuntimeError("No model outputs found. Set RUN_MODELS=True on a CUDA GPU.")
OK=OUTPUTS[OUTPUTS.status=="OK"].copy()
CASE_META=APPROVED.copy()
MERGED=OK.merge(CASE_META,on=["case_id","thread_id"],how="left",suffixes=("","_case"))
STOP={"the","a","an","and","or","is","are","to","of","in","for","with","de","la","le","les","un","une","et","ou","ki","li","mo","to","so","nou","zot","sa"}
def diagnostics(r):
    context=str(r.get("supplied_context") or ""); output=str(r.get("raw_output") or ""); gold=str(r.get("effective_gold_answer") or "")
    cwords=set(tokenize_words(context)); owords=[w for w in tokenize_words(output) if w not in STOP]
    unsupported_words=[w for w in owords if len(w)>2 and w not in cwords and not w.startswith("person_")]
    cnums=set(re.findall(r"\b\d+(?:[.,]\d+)?\b",context)); onums=set(re.findall(r"\b\d+(?:[.,]\d+)?\b",output))
    cpersons=set(re.findall(r"PERSON_\d+",context)); opersons=set(re.findall(r"PERSON_\d+",output))
    cturns=set(re.findall(r"\b(?:C|R)\d+\b",context)); oturns=set(re.findall(r"\b(?:C|R)\d+\b",output))
    try: json.loads(output); valid_json=True
    except Exception: valid_json=False
    abstain=bool(re.search(r"INSUFFICIENT[_ ]EVIDENCE|LINK_CONTENT_NOT_SUPPLIED|cannot (?:be )?determined|not enough (?:information|evidence)",output,re.I))
    expected=str(r.get("effective_answerability") or "").upper()
    expected_insufficient=any(x in expected for x in ["INSUFFICIENT","UNANSWERABLE","NOT_SUPPLIED"])
    grounding=1-(len(unsupported_words)/max(1,len(owords)))
    return pd.Series({"json_valid":valid_json,"abstained":abstain,"expected_insufficient":expected_insufficient,"abstention_correct":abstain==expected_insufficient,"unsupported_number_count":len(onums-cnums),"unsupported_placeholder_count":len(opersons-cpersons),"unsupported_turn_label_count":len(oturns-cturns),"lexical_grounding_ratio":grounding,"gold_token_f1":token_f1(output,gold) if gold not in {"","None","nan"} else np.nan,"rule_flag_unsupported":bool((onums-cnums) or (opersons-cpersons) or (oturns-cturns)),"rule_flag_format":not valid_json})
DIAG=MERGED.apply(diagnostics,axis=1)
EVALUATED=pd.concat([MERGED.reset_index(drop=True),DIAG.reset_index(drop=True)],axis=1)
if not TOKENS.empty:
    us=TOKENS.groupby(["case_id","model_id"]).agg(mean_entropy=("entropy","mean"),max_entropy=("entropy","max"),mean_logprob=("logprob","mean"),min_logprob=("logprob","min")).reset_index()
    EVALUATED=EVALUATED.merge(us,on=["case_id","model_id"],how="left")
EVALUATED.to_parquet(OUTPUT_DIR/"automatic_evaluation.parquet",index=False)
print(EVALUATED[["model_id","task_family","json_valid","abstention_correct","rule_flag_unsupported","lexical_grounding_ratio"]].head())

## 13. Automated provisional semantic judge — no human gate

In [14]:
JUDGE_LABELS=["SUPPORTED_CORRECT","UNSUPPORTED_ADDITION","CONTRADICTS_EVIDENCE","FABRICATED_ENTITY","FABRICATED_ATTRIBUTE_RELATION","SOURCE_OVERREACH","UNJUSTIFIED_RESOLUTION","ANSWERABILITY_FAILURE","WRONG_EXISTING_ITEM","OMISSION","MISINTERPRETATION","LANGUAGE_OR_ORTHOGRAPHY_DISTORTION","STRUCTURE_ERROR","FORMAT_ERROR","PRIVACY_FAILURE","REFUSAL_OR_NONANSWER","AMBIGUOUS_OR_UNGRADABLE"]
def judge_prompt(r):
    return f'''You are an evidence-grounding evaluator. Evaluate the MODEL RESPONSE only against the supplied CONVERSATION, TASK, and GOLD INFORMATION. Do not use external knowledge. Return one JSON object with keys: hallucination (true/false), supported (true/false), primary_label, secondary_labels (list), confidence (0 to 1), unsupported_spans (list), brief_reason. Allowed labels: {JUDGE_LABELS}. If evidence or gold data is genuinely insufficient, use AMBIGUOUS_OR_UNGRADABLE.

CONVERSATION:
{r.get('supplied_context')}

TASK:
{r.get('question_or_instruction')}

GOLD INFORMATION:
answerability={r.get('effective_answerability')}
evidence={r.get('effective_evidence')}
answer={r.get('effective_gold_answer')}

MODEL RESPONSE:
{r.get('raw_output')}'''
def parse_json_object(text):
    text=re.sub(r"<think>.*?</think>","",str(text),flags=re.S).strip()
    try: return json.loads(text)
    except Exception:
        m=re.search(r"\{.*\}",text,re.S)
        if m:
            try:return json.loads(m.group(0))
            except Exception:pass
        return {"hallucination":None,"supported":None,"primary_label":"AMBIGUOUS_OR_UNGRADABLE","secondary_labels":[],"confidence":0,"unsupported_spans":[],"brief_reason":"Judge output could not be parsed.","raw_judge_output":text}
JUDGE_PATH=OUTPUT_DIR/"automated_judgments.jsonl"
if RUN_AUTOMATED_JUDGE:
    existing_j=load_jsonl(JUDGE_PATH); done={(x.get("case_id"),x.get("generator_model_id")) for x in existing_j if x.get("status")=="OK"}
    judge_id=MODEL_REGISTRY[JUDGE_MODEL_KEY]["model_id"]
    print("Loading provisional judge:",judge_id)
    jt,jm=load_bundle(judge_id)
    try:
        for n,(_,r) in enumerate(EVALUATED.iterrows(),1):
            sig=(r["case_id"],r["model_id"])
            if RESUME and sig in done: continue
            messages=[{"role":"user","content":judge_prompt(r)}]
            rendered=jt.apply_chat_template(messages,tokenize=False,add_generation_prompt=True) if getattr(jt,"chat_template",None) else messages[0]["content"]+"\nJSON:"
            enc=jt(rendered,return_tensors="pt",truncation=True,max_length=MAX_INPUT_TOKENS); device=next(jm.parameters()).device; enc={k:v.to(device) for k,v in enc.items()}
            with torch.inference_mode(): out=jm.generate(**enc,max_new_tokens=256,do_sample=False,pad_token_id=jt.pad_token_id)
            text=jt.decode(out[0,enc["input_ids"].shape[1]:],skip_special_tokens=True); parsed=parse_json_object(text)
            record={"status":"OK","case_id":r["case_id"],"thread_id":r["thread_id"],"generator_model_id":r["model_id"],"judge_model_id":judge_id,"judge_type":"AUTOMATED_PROVISIONAL_NOT_GOLD",**parsed}
            append_jsonl(JUDGE_PATH,record)
            if n%25==0 or n==len(EVALUATED): print("judged",n,"/",len(EVALUATED))
    finally: unload_bundle(jt,jm)
else: print("RUN_AUTOMATED_JUDGE=False")

judged 75 / 3804
judged 100 / 3804
judged 125 / 3804
judged 150 / 3804
judged 175 / 3804
judged 200 / 3804
judged 225 / 3804
judged 250 / 3804
judged 275 / 3804
judged 300 / 3804
judged 325 / 3804
judged 350 / 3804
judged 375 / 3804
judged 400 / 3804
judged 425 / 3804
judged 450 / 3804
judged 475 / 3804
judged 500 / 3804
judged 525 / 3804
judged 550 / 3804
judged 575 / 3804
judged 600 / 3804
judged 625 / 3804
judged 650 / 3804
judged 675 / 3804
judged 700 / 3804
judged 725 / 3804
judged 750 / 3804
judged 775 / 3804
judged 800 / 3804
judged 825 / 3804
judged 850 / 3804
judged 875 / 3804
judged 900 / 3804
judged 925 / 3804
judged 950 / 3804
judged 975 / 3804
judged 1000 / 3804
judged 1025 / 3804
judged 1050 / 3804
judged 1075 / 3804
judged 1100 / 3804
judged 1125 / 3804
judged 1150 / 3804
judged 1175 / 3804
judged 1200 / 3804
judged 1225 / 3804
judged 1250 / 3804
judged 1275 / 3804
judged 1300 / 3804
judged 1325 / 3804
judged 1350 / 3804
judged 1375 / 3804
judged 1400 / 3804
judged 1425 

## 14. Merge categories and create model/case comparisons

In [15]:
JUDGED=pd.DataFrame(load_jsonl(JUDGE_PATH)) if JUDGE_PATH.exists() else pd.DataFrame()
FINAL=EVALUATED.copy()
if not JUDGED.empty:
    FINAL=FINAL.merge(JUDGED[["case_id","generator_model_id","hallucination","supported","primary_label","secondary_labels","confidence","unsupported_spans","brief_reason"]],left_on=["case_id","model_id"],right_on=["case_id","generator_model_id"],how="left")
FINAL.to_parquet(OUTPUT_DIR/"categorized_model_outputs.parquet",index=False)
model_summary=FINAL.groupby("model_id").agg(outputs=("case_id","count"),json_valid_rate=("json_valid","mean"),abstention_accuracy=("abstention_correct","mean"),unsupported_rule_rate=("rule_flag_unsupported","mean"),mean_grounding=("lexical_grounding_ratio","mean"),mean_entropy=("mean_entropy","mean") if "mean_entropy" in FINAL else ("case_id","count"),provisional_hallucination_rate=("hallucination","mean") if "hallucination" in FINAL else ("case_id","count")).reset_index()
model_summary.to_csv(OUTPUT_DIR/"model_summary.csv",index=False)
if "primary_label" in FINAL:
    pd.crosstab(FINAL.model_id,FINAL.primary_label).to_csv(OUTPUT_DIR/"error_category_by_model.csv")
completion=pd.crosstab(FINAL.case_id,FINAL.model_id).reindex(columns=[MODEL_REGISTRY[k]["model_id"] for k in MODEL_KEYS],fill_value=0)
completion["all_four_complete"]=(completion>0).all(axis=1)
completion.to_csv(OUTPUT_DIR/"paired_case_completeness.csv")
display(model_summary)

,model_id,outputs,json_valid_rate,abstention_accuracy,unsupported_rule_rate,mean_grounding,mean_entropy,provisional_hallucination_rate
0,HuggingFaceTB/SmolLM3-3B,951,0.000000,0.932702,0.086225,0.400166,0.401926,0.131855
1,Qwen/Qwen3-4B-Instruct-2507,951,0.983176,0.996845,0.128286,0.449780,0.064902,0.017076
2,Qwen/Qwen3-4B-Thinking-2507,951,0.000000,0.854890,0.202944,0.481669,0.221725,0.0
3,microsoft/Phi-4-mini-instruct,951,0.896951,0.970557,0.126183,0.449148,0.297392,0.073955


## 15. Candidate benchmark-dimension induction

In [16]:
DIMENSION_FIELDS=["task_class","task_family","task_variant","stratum","cand_language_profile","validated_language_profile","validated_code_switching","reference_ambiguity","exophoric_reference","local_cultural_dependence","evidence_sufficiency","fragmentary_confirmed","ambiguity_flag","answerability","has_url","has_placeholder","placeholder_recurrence","has_emoji","expressive_punctuation","repeated_characters"]
dimension_rows=[]
for field in DIMENSION_FIELDS:
    if field not in FINAL.columns: continue
    for value,g in FINAL.groupby(field,dropna=False):
        value_text="MISSING" if pd.isna(value) else str(value)
        rest=FINAL[~FINAL.index.isin(g.index)]
        h_rate=float(g.hallucination.dropna().mean()) if "hallucination" in g and g.hallucination.notna().any() else np.nan
        r_rate=float(rest.hallucination.dropna().mean()) if "hallucination" in rest and rest.hallucination.notna().any() else np.nan
        labels=g.primary_label.value_counts().to_dict() if "primary_label" in g else {}
        dimension_rows.append({"dimension_field":field,"dimension_value":value_text,"outputs":len(g),"unique_cases":g.case_id.nunique(),"unique_threads":g.thread_id.nunique(),"unique_posts":g.post_id.nunique() if "post_id" in g else np.nan,"models":g.model_id.nunique(),"provisional_hallucination_rate":h_rate,"comparison_rate_without_feature":r_rate,"rate_difference":h_rate-r_rate if pd.notna(h_rate) and pd.notna(r_rate) else np.nan,"json_failure_rate":float((~g.json_valid).mean()),"abstention_accuracy":float(g.abstention_correct.mean()),"unsupported_rule_rate":float(g.rule_flag_unsupported.mean()),"mean_grounding":float(g.lexical_grounding_ratio.mean()),"mean_entropy":float(g.mean_entropy.mean()) if "mean_entropy" in g else np.nan,"failure_labels_json":json.dumps(labels,ensure_ascii=False),"status":"PROMISING_FOR_CONFIRMATION" if len(g)>=20 and g.thread_id.nunique()>=10 and pd.notna(h_rate) and abs(h_rate-r_rate)>=0.10 else "OBSERVED_INPUT_FEATURE","researcher_decision":"","researcher_rationale":"","confirmation_status":""})
DIMENSIONS=pd.DataFrame(dimension_rows).sort_values(["status","rate_difference"],ascending=[True,False])
DIMENSIONS.to_csv(OUTPUT_DIR/"candidate_dimension_matrix.csv",index=False)
DIMENSIONS.to_excel(OUTPUT_DIR/"candidate_dimension_matrix.xlsx",index=False)
print("Promising provisional candidates:")
display(DIMENSIONS[DIMENSIONS.status=="PROMISING_FOR_CONFIRMATION"].head(30))

Promising provisional candidates:


,dimension_field,dimension_value,outputs,unique_cases,unique_threads,unique_posts,models,provisional_hallucination_rate,comparison_rate_without_feature,rate_difference,json_failure_rate,abstention_accuracy,unsupported_rule_rate,mean_grounding,mean_entropy,failure_labels_json,status,researcher_decision,researcher_rationale,confirmation_status


## 16. Error-pattern and uncertainty analyses

In [17]:
analysis=[]
for keys,g in FINAL.groupby(["model_id","task_family"],dropna=False):
    analysis.append({"model_id":keys[0],"task_family":keys[1],"outputs":len(g),"hallucination_rate":float(g.hallucination.dropna().mean()) if "hallucination" in g and g.hallucination.notna().any() else np.nan,"json_failure_rate":float((~g.json_valid).mean()),"abstention_accuracy":float(g.abstention_correct.mean()),"unsupported_rule_rate":float(g.rule_flag_unsupported.mean()),"mean_entropy":float(g.mean_entropy.mean()) if "mean_entropy" in g else np.nan,"mean_logprob":float(g.mean_logprob.mean()) if "mean_logprob" in g else np.nan})
ANALYSIS=pd.DataFrame(analysis)
ANALYSIS.to_csv(OUTPUT_DIR/"failure_by_model_and_task.csv",index=False)
if "primary_label" in FINAL:
    labels=FINAL.groupby(["task_family","primary_label"]).size().reset_index(name="count")
    labels.to_csv(OUTPUT_DIR/"failure_label_by_task.csv",index=False)
# Paired case disagreement across generators
paired=FINAL.pivot_table(index="case_id",columns="model_id",values="hallucination",aggfunc="first") if "hallucination" in FINAL else pd.DataFrame()
if not paired.empty:
    paired["models_with_label"]=paired.notna().sum(axis=1)
    paired["hallucination_votes"]=paired.fillna(False).sum(axis=1)
    paired["cross_model_disagreement"]=paired.drop(columns=["models_with_label","hallucination_votes"]).nunique(axis=1)>1
    paired.to_csv(OUTPUT_DIR/"cross_model_case_disagreement.csv")
display(ANALYSIS.head(30))

,model_id,task_family,outputs,hallucination_rate,json_failure_rate,abstention_accuracy,unsupported_rule_rate,mean_entropy,mean_logprob
0,HuggingFaceTB/SmolLM3-3B,answerability_abstention,240,0.004219,1.000000,0.745833,0.150000,0.365798,-0.164810
1,HuggingFaceTB/SmolLM3-3B,attribution,109,0.201835,1.000000,1.000000,0.073394,0.292804,-0.135243
2,HuggingFaceTB/SmolLM3-3B,conversational_grounding,59,0.237288,1.000000,1.000000,0.000000,0.584098,-0.232293
3,HuggingFaceTB/SmolLM3-3B,evidence_grounded_qa,107,0.000000,1.000000,1.000000,0.280374,0.363533,-0.169279
4,HuggingFaceTB/SmolLM3-3B,evidence_supported_inference,43,0.309524,1.000000,1.000000,0.000000,0.581299,-0.239898
5,HuggingFaceTB/SmolLM3-3B,exact_list_extraction,60,0.051724,1.000000,1.000000,0.033333,0.323710,-0.144165
6,HuggingFaceTB/SmolLM3-3B,explicit_claim_extraction,96,0.182927,1.000000,1.000000,0.010417,0.222922,-0.090378
7,HuggingFaceTB/SmolLM3-3B,faithful_summarization,49,0.304348,1.000000,1.000000,0.020408,0.714981,-0.299559
8,HuggingFaceTB/SmolLM3-3B,missing_context_abstention,3,0.000000,1.000000,0.000000,0.000000,0.434538,-0.185327
9,HuggingFaceTB/SmolLM3-3B,placeholder_continuity,34,0.303030,1.000000,1.000000,0.029412,0.107022,-0.050724


## 17. Optional matched hidden-state analysis

In [18]:
if RUN_HIDDEN_STATE_ANALYSIS:
    if "hallucination" not in FINAL: raise RuntimeError("Automated provisional judgments are required before matched hidden-state analysis.")
    selected_pairs=[]
    for model_id,g in FINAL.groupby("model_id"):
        bad=g[g.hallucination==True].sort_values("confidence",ascending=False).head(10)
        good=g[g.supported==True].sort_values("confidence",ascending=False).head(len(bad))
        selected_pairs.extend([(model_id,x,"hallucinated") for x in bad.case_id])
        selected_pairs.extend([(model_id,x,"supported_control") for x in good.case_id])
    hs_rows=[]
    for model_id in sorted({x[0] for x in selected_pairs}):
        tok,model=load_bundle(model_id)
        try:
            model_cases=[x for x in selected_pairs if x[0]==model_id]
            for _,case_id,label in model_cases:
                r=FINAL[(FINAL.model_id==model_id)&(FINAL.case_id==case_id)].iloc[0]
                messages=make_messages(r); rendered=tok.apply_chat_template(messages,tokenize=False,add_generation_prompt=True) if getattr(tok,"chat_template",None) else SYSTEM_PROMPT+"\n\n"+messages[-1]["content"]
                full=rendered+str(r.raw_output); enc=tok(full,return_tensors="pt",truncation=True,max_length=MAX_INPUT_TOKENS); device=next(model.parameters()).device; enc={k:v.to(device) for k,v in enc.items()}
                with torch.inference_mode(): out=model(**enc,output_hidden_states=True,use_cache=False)
                layers=out.hidden_states; indices=sorted(set([0,len(layers)//2,len(layers)-1]))
                for li in indices:
                    v=layers[li][0,-1].float().cpu().numpy()
                    hs_rows.append({"model_id":model_id,"case_id":case_id,"match_label":label,"layer_index":li,"hidden_dim":len(v),"l2_norm":float(np.linalg.norm(v)),"mean":float(v.mean()),"std":float(v.std())})
        finally: unload_bundle(tok,model)
    pd.DataFrame(hs_rows).to_csv(OUTPUT_DIR/"matched_hidden_state_summaries.csv",index=False)
else: print("Hidden-state analysis disabled. Set RUN_HIDDEN_STATE_ANALYSIS=True after outputs are categorized.")

Hidden-state analysis disabled. Set RUN_HIDDEN_STATE_ANALYSIS=True after outputs are categorized.


## 18. Visualizations and methodology summary

In [19]:
import matplotlib.pyplot as plt
if not ANALYSIS.empty:
    plot=ANALYSIS.pivot(index="task_family",columns="model_id",values="hallucination_rate")
    ax=plot.plot(kind="bar",figsize=(14,6)); ax.set_title("Provisional automated hallucination rate by model and task"); ax.set_ylabel("Rate"); ax.set_xlabel("Task family"); plt.xticks(rotation=45,ha="right"); plt.tight_layout(); plt.savefig(OUTPUT_DIR/"hallucination_by_model_task.png",dpi=180); plt.close()
summary=f'''# Pilot methodology and outputs

- Input: already reviewed workbook `{INPUT_XLSX.name}`.
- Approved cases loaded: {len(APPROVED)}.
- Generators: {', '.join(MODEL_REGISTRY[k]['model_id'] for k in MODEL_KEYS)}.
- Models receive the same case set and evidence.
- Outputs, token probabilities, log probabilities and entropy are stored.
- Rule-based checks identify formatting, abstention and unsupported identifiers.
- The automated semantic judge produces provisional labels only; it is not human gold.
- Candidate dimensions are produced by relating input/task characteristics to provisional failure categories.
- No dimension is automatically final. Promising candidates require later confirmation.
'''
(OUTPUT_DIR/"methodology_summary.md").write_text(summary,encoding="utf-8")
manifest={"notebook_version":NOTEBOOK_VERSION,"run_utc":datetime.now(timezone.utc).isoformat(),"input_sha256":sha256_file(INPUT_XLSX),"approved_cases":len(APPROVED),"models":[MODEL_REGISTRY[k] for k in MODEL_KEYS],"load_strategy":EFFECTIVE_LOAD_STRATEGY,"run_models":RUN_MODELS,"run_automated_judge":RUN_AUTOMATED_JUDGE,"case_limit":CASE_LIMIT,"outputs":[p.name for p in OUTPUT_DIR.iterdir()]}
(OUTPUT_DIR/"run_manifest.json").write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
print(summary)

# Pilot methodology and outputs

- Input: already reviewed workbook `pilot_gold_review_completed.xlsx`.
- Approved cases loaded: 951.
- Generators: Qwen/Qwen3-4B-Instruct-2507, Qwen/Qwen3-4B-Thinking-2507, microsoft/Phi-4-mini-instruct, HuggingFaceTB/SmolLM3-3B.
- Models receive the same case set and evidence.
- Outputs, token probabilities, log probabilities and entropy are stored.
- Rule-based checks identify formatting, abstention and unsupported identifiers.
- The automated semantic judge produces provisional labels only; it is not human gold.
- Candidate dimensions are produced by relating input/task characteristics to provisional failure categories.
- No dimension is automatically final. Promising candidates require later confirmation.



## 19. What to inspect after execution

The main outputs are:

- `model_outputs.jsonl`: raw outputs from every generator;
- `token_uncertainty.jsonl`: token probability, log probability and entropy;
- `automated_judgments.jsonl`: provisional semantic failure labels;
- `categorized_model_outputs.parquet`: combined cases, outputs, diagnostics and labels;
- `model_summary.csv`: overall comparison;
- `failure_by_model_and_task.csv`: performance by task;
- `failure_label_by_task.csv`: observed error types;
- `cross_model_case_disagreement.csv`: cases where models behave differently;
- `candidate_dimension_matrix.xlsx`: characteristics associated with failure patterns;
- `methodology_summary.md` and `run_manifest.json`: reproducibility record.

No additional human review file is required for this notebook to run.